# 03. HARFM Dataset — Exploratory Data Analysis

This notebook provides a comprehensive EDA of the final **HARFM** dataset.

## Input
`HARFM.csv` — output of `02_generation.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"]      = 11
sns.set_style("whitegrid")

DATA_PATH = Path(".") / "HARFM.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
df.head()

## 1. Basic Info & Missing Values

In [ ]:
print("=== dtypes ===")
print(df.dtypes)
print("\n=== Missing values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"count": missing, "pct(%)": missing_pct})[missing > 0].sort_values("count", ascending=False)

## 2. Label Distribution (2-way & 4-way)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["2_way_label"].value_counts().plot(
    kind="bar", ax=axes[0], color=["#2ecc71", "#e74c3c"])
axes[0].set_title("2-way label (real / fake)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df["4_way_label"].value_counts().plot(
    kind="bar", ax=axes[1], color=["#3498db", "#9b59b6", "#e74c3c", "#f39c12"])
axes[1].set_title("4-way label (HR / AR / HF / AF)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("4-way counts:")
print(df["4_way_label"].value_counts())
print("\n4-way ratios:")
print((df["4_way_label"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")

## 3. Numeric Variable Summary

In [ ]:
num_cols = [c for c in ["score", "num_comments", "title_len", "upvote_ratio"] if c in df.columns]
print(df[num_cols].describe())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=50, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

## 4. Numeric Variables by 4-way Label

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(5 * len(num_cols), 5))
if len(num_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, num_cols):
    df.boxplot(column=col, by="4_way_label", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Headline Length Distribution by Label

In [ ]:
if "title_len" not in df.columns:
    df["title_len"] = df["clean_title"].astype(str).str.len()

fig, ax = plt.subplots(figsize=(10, 5))
for label, grp in df.groupby("4_way_label"):
    grp["title_len"].hist(bins=50, alpha=0.5, label=label, ax=ax)
ax.set_xlabel("Headline length (chars)")
ax.set_ylabel("Count")
ax.set_title("Headline length distribution by 4-way label")
ax.legend()
plt.tight_layout()
plt.show()

print(df.groupby("4_way_label")["title_len"].describe().round(1))

## 6. AF Strategy Distribution

In [ ]:
if "ai_strategy" in df.columns:
    af_strat = df[df["4_way_label"] == "AF"]["ai_strategy"].value_counts()
    af_strat.plot(kind="bar", figsize=(8, 4), color="#f39c12", edgecolor="black")
    plt.title("AF strategy distribution")
    plt.xlabel("Strategy")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    print(af_strat)

## 7. Top Subreddits

In [ ]:
top_n = 20
top_subs = df["subreddit"].value_counts().head(top_n)
top_subs.plot(kind="barh", figsize=(10, 6))
plt.title(f"Top {top_n} subreddits")
plt.xlabel("Count")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Temporal Distribution

In [ ]:
if "created_utc" in df.columns:
    df["year"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce").dt.year
    year_counts = df.groupby(["year", "4_way_label"]).size().unstack(fill_value=0)
    year_counts.plot(kind="bar", figsize=(12, 5), stacked=True)
    plt.title("Posts per year by 4-way label")
    plt.xlabel("Year")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 9. Sample Headlines per Label

In [ ]:
for label in ["HR", "HF", "AR", "AF"]:
    print(f"\n{'='*60}")
    print(f"  {label} — 5 random samples")
    print(f"{'='*60}")
    subset = df[df["4_way_label"] == label].sample(5, random_state=42)
    for _, row in subset.iterrows():
        print(f"  [original]  {row['clean_title']}")
        if label in ["AR", "AF"]:
            print(f"  [generated] {row['final_headline']}")
        print()